In [94]:
import mne
import numpy as np
from numpy import pi
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score
from scipy.signal import welch

# quantum comp sci
import pennylane as qml
from pennylane import numpy as pnp

In [95]:
p1 = mne.io.read_raw('../../data/1/1.edf', preload=True)
a1 = pd.read_csv('../../data/1/1_1.txt', header=None)

Extracting EDF parameters from ../../data/1/1.edf...
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 5279999  =      0.000 ... 26399.995 secs...


In [96]:
print(f"Channels p1: {p1.ch_names}") # C4 is in

Channels p1: ['LOC-A2', 'ROC-A1', 'F3-A2', 'C3-A2', 'O1-A2', 'F4-A1', 'C4-A1', 'O2-A1', 'X1', 'X2', 'X3', 'X4', 'X5', 'X6', 'DC3', 'X7', 'X8', 'SaO2', 'DC8']


In [97]:
print(f"Unique annotations: {a1[0].unique()}")

Unique annotations: [0 1 2 3 5]


In [98]:
# check length of 
def print_len_match(raw, annot):
    fs = raw.info["sfreq"]
    print(f"Length: {raw.n_times / fs / len(annot)}")

print_len_match(p1, a1)

Length: 30.0


In [99]:
def load_subject_for_training(edf_path, label_path, channel="C4-A1"):
    raw = mne.io.read_raw(edf_path, preload=True)
    fs = int(raw.info["sfreq"])
    signal = raw.get_data(picks=channel)[0]
    labels = pd.read_csv(label_path, header=None).values.reshape(-1)
    epoch_length = 30

    epochs = signal.reshape(len(labels), fs * epoch_length)
    assert len(signal) == len(labels) * fs * epoch_length
    return epochs, labels, fs

p1, a1, fs1 = load_subject_for_training('../../data/1/1.edf', '../../data/1/1_1.txt')

Extracting EDF parameters from ../../data/1/1.edf...
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 5279999  =      0.000 ... 26399.995 secs...


In [100]:
print(p1.shape, np.bincount(a1))

(880, 6000) [264  73 194 231   0 118]


In [101]:
# working with sleep frequency bands
# delta =    0.5 - 4     |   deep sleep (N3). Large, slow waves
# theta =    4-8         |   drowsiness, N1, REM
# alpha =    8-12        |   relaxed wakefulness with eyes closed, fades as we fall asleep
# sigma =    12-16       |   N2 spindle area

BANDS = {
    "delta" : (0.5, 4),     # N3 marker: slow waves of 6s to score block as N3
    "theta" : (4, 8),       # fast waves 
    "alpha" : (8, 12),      # fast waves
    "sigma" : (12, 16),     # N2 marker: slow waves with spindles
}

def band_features(epochs, fs):
    # power spectrum per epoch: 4s windows, 50% overlap, averaged
    freqs, psd = welch(epochs, fs=fs, nperseg=4 * fs, axis = -1) # psd: (n_epochs, 401)
    
    feats = []
    for lo, hi in BANDS.values():
        in_band = (freqs >= lo) & (freqs < hi) # mask over the frequencies
        feats.append(psd[:, in_band].mean(axis=1)) # mean power in band per epoch

    # return log for outliers - one outlier could skew everyting
    return np.log(np.stack(feats, axis=1)) # (n_epochs, 4)

In [102]:
F = band_features(p1, fs1)
print(F.shape)
for i, name in enumerate(BANDS):
    print(f"{name}, N2 : {F[a1 == 2, i].mean():.2f} || N3 : {F[a1 == 3, i].mean():.2f}")

(880, 4)
delta, N2 : -28.40 || N3 : -27.52
theta, N2 : -29.92 || N3 : -29.83
alpha, N2 : -30.68 || N3 : -30.83
sigma, N2 : -31.43 || N3 : -31.53


In [103]:
def prep(edf, txt):
    epochs, labels, fs = load_subject_for_training(edf, txt)
    keep = np.isin(labels, [2,3]) # we want N2 and N3 only for this experiment
    X = band_features(epochs[keep], fs) # keep only N2 adn N3
    y = (labels[keep] == 3).astype(int) # annotate N2 = false and N3 = true
    return X, y

Xa, ya = prep('../../data/1/1.edf', '../../data/1/1_1.txt')
Xb, yb = prep('../../data/2/2.edf', '../../data/2/2_1.txt')
X_test, y_test = prep('../../data/3/3.edf', '../../data/3/3_1.txt')

X_train, y_train = np.vstack([Xa, Xb]), np.concatenate([ya, yb])
print(f"Train: {np.bincount(y_train)}, Test: {np.bincount(y_test)}")

sc = MinMaxScaler(feature_range=(0, np.pi)).fit(X_train) # define scaler map every band between [0, pi] reason is: we are doing rotations (flip our data)
X_train = sc.transform(X_train) # scale the values
X_test = np.clip(sc.transform(X_test), 0, np.pi) # capping below 0 and above pi to keep the rotation in tact

clf = LogisticRegression().fit(X_train, y_train) 
print(f"Logistic Regression balanced accuracy: {balanced_accuracy_score(y_test, clf.predict(X_test))}")

Extracting EDF parameters from ../../data/1/1.edf...
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 5279999  =      0.000 ... 26399.995 secs...
Extracting EDF parameters from ../../data/2/2.edf...
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 5783999  =      0.000 ... 28919.995 secs...
Extracting EDF parameters from ../../data/3/3.edf...
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 5657999  =      0.000 ... 28289.995 secs...
Train: [539 388], Test: [246 173]
Logistic Regression balanced accuracy: 0.8427557685981484


In [104]:
clf_delta = LogisticRegression().fit(X_train[:, [0]], y_train)
print(f"delta only: {balanced_accuracy_score(y_test, clf_delta.predict(X_test[:, [0]]))}")
# delta waves are the most important waves... the other basically just add noise. the task comes down to a 1 variable check

delta only: 0.8592391559753748


What is a Qubit?
a bit is 0 and 1, a qubit is measured by 2 numbers, alpha and beta: how much 0 and how much 1 we have, written as a|0> + b|1>
- measuring gives 0 with probability a^2 and 1 with probability b^2 (=1)
- after measuring qubit is always 0 or 1. the inbetween state -> how much of each it is -> is gone (if you look you always know the result)
- picture: sphere with north pole = 0 and south pole = 1 and we have an arrow pointing in some direction = the superposition

what does a rotation do? 
- A gate chanes alpha and beta. RY(theta) turns the arrow by angle theta
RY(theta)|0> = cos(theta/2)|0> + sin(theta/2)|1>, so P(1) = sin^2(theta/2)

- theta = 0 leaves it at 0
- theta = pi/2 gives a 50/50
- theta = pi gives 1
-> taht is what angle embeddings does
-> delta power of 2.01 tilts qubits 0 by 2.01 radians (since it is more than pi/2 it will eventually end up at 1 with enough runs)

what is a CNOT?
basically a NOT gate but the quantum edition. it's a two-qubit gate: "if control = 1 flip the target."
- 0,0 -> 0,0
- 1,0 -> 1,1

In [145]:
n_qubits, n_layers = 4, 2 # we have 4 qubits for our 4 freq. bands, 2 layers (like in a neural network)
dev = qml.device("default.qubit", wires=n_qubits) # exact simulator, torch.device("cpu") quantum edition

@qml.qnode(dev) # -> basically define a torch.nn.Module and the circuit function is the forward function, difference is we store the weights separately
def circuit(weights, x):
    # AngleEmbedding = input layer (similar to an input layer in a neural network)
    # we rotate the qubits
    qml.AngleEmbedding(x, wires=range(n_qubits), rotation="Y") # feature i -> RY(x_i) on qubit i
    # StronglyEntanglingLayers = hidden layer -> we give every qubit 3 trainable rotation angles
    # rotations = per-qubit weights, CNOTs = connections between qubits. 
    # 2*4*3 = 24 trainable weights
    # the CNOTs happen in this strongly entangling layer
    qml.StronglyEntanglingLayers(weights, wires=range(n_qubits)) # trainable rotations + CNOTs
    # we measure the qubits
    return qml.expval(qml.PauliZ(0)) # read qubit 0 -> value in [-1, 1]

In [151]:
from qiskit_ibm_runtime.fake_provider import FakeManilaV2

backend = FakeManilaV2()   # 5-qubit noise model taken from a real IBM device
dev_fake = qml.device("qiskit.remote", wires=5, backend=backend, shots=1000, seed_transpiler=42)

@qml.qnode(dev_fake)
def circuit_hw(weights, x):          # same body as circuit, just a different device
    qml.AngleEmbedding(x, wires=range(4), rotation="Y")
    qml.StronglyEntanglingLayers(weights, wires=range(4))
    return qml.expval(qml.PauliZ(0))

c:\Users\danie\vscode\quantum_computing_101\quantum-sleep\.venv\Lib\site-packages\pennylane\devices\device_api.py:207: PennyLaneDeprecationWarning: Setting shots on device is deprecated. Please use the `set_shots` transform on the respective QNode instead.
  warnings.warn(


In [152]:
shape = qml.StronglyEntanglingLayers.shape(n_layers=n_layers, n_wires=n_qubits)  # (2, 4, 3)
print(shape)
print(qml.draw(circuit, level="device")(pnp.zeros(shape), X_train[0]))

(2, 4, 3)
0: ──RY(2.01)──Rot(0.00,0.00,0.00)─╭●───────╭X──Rot(0.00,0.00,0.00)─╭●────╭X────┤  <Z>
1: ──RY(2.73)──Rot(0.00,0.00,0.00)─╰X─╭●────│───Rot(0.00,0.00,0.00)─│──╭●─│──╭X─┤     
2: ──RY(2.94)──Rot(0.00,0.00,0.00)────╰X─╭●─│───Rot(0.00,0.00,0.00)─╰X─│──╰●─│──┤     
3: ──RY(1.78)──Rot(0.00,0.00,0.00)───────╰X─╰●──Rot(0.00,0.00,0.00)────╰X────╰●─┤     


In [153]:
def model(weights, bias, X):
    return circuit(weights, X) + bias # we can pass everything at once - closed form solution style

def loss(weights, bias, X, t):
    return pnp.mean((t - model(weights, bias, X)) ** 2) # use mse quantum edition

t_train = 2 * y_train - 1 # targets: N2 -> -1, N3 -> +1

In [155]:
def predict(weights, bias, X):
    return (np.asarray(model(weights, bias, X)) > 0).astype(int)


def train_vqc(seed, steps=100, lr=0.05):
    pnp.random.seed(seed)
    weights = pnp.random.uniform(0, 2 * pnp.pi, size=shape, requires_grad=True)
    bias = pnp.array(0.0, requires_grad=True)
    opt = qml.AdamOptimizer(stepsize=lr)
    rng = np.random.default_rng(seed)
    for _ in range(steps):
        idx = rng.choice(len(X_train), size=64, replace=False)
        (weights, bias), _ = opt.step_and_cost(loss, weights, bias, X=X_train[idx], t=t_train[idx])
    return weights, bias                      # return the trained model, not a score


def evaluate(circ, weights, bias, X, y):
    pred = (np.asarray(circ(weights, X)) + bias > 0).astype(int)
    return balanced_accuracy_score(y, pred)

results = []
for s in [4, 42, 67, 69, 221]:
    w, b = train_vqc(s)
    results.append([evaluate(circuit, w, b, X_test, y_test),       # ideal
                    evaluate(circuit_hw, w, b, X_test, y_test)])   # fake IBM chip
results = np.array(results)

In [156]:
from qiskit_ibm_runtime import QiskitRuntimeService
from dotenv import load_dotenv
import os

load_dotenv()

QiskitRuntimeService.save_account(channel="ibm_quantum_platform",
    token=os.environ["ibm"], instance=os.environ["crn"], overwrite=True)

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)
print(backend.name, backend.num_qubits)

dev_ibm = qml.device("qiskit.remote", wires=backend.num_qubits, backend=backend, shots=1000)

@qml.qnode(dev_ibm)
def circuit_ibm(weights, x):
    qml.AngleEmbedding(x, wires=range(4), rotation="Y")
    qml.StronglyEntanglingLayers(weights, wires=range(4))
    return qml.expval(qml.PauliZ(0))

ibm_kingston 156


c:\Users\danie\vscode\quantum_computing_101\quantum-sleep\.venv\Lib\site-packages\pennylane\devices\device_api.py:207: PennyLaneDeprecationWarning: Setting shots on device is deprecated. Please use the `set_shots` transform on the respective QNode instead.
  warnings.warn(


In [157]:
rng = np.random.default_rng(0)
sub = np.concatenate([rng.choice(np.where(y_test == c)[0], 20, replace=False) for c in (0, 1)])
X_sub, y_sub = X_test[sub], y_test[sub]

w, b = train_vqc(42)
print(backend.name)
for name, circ in [("ideal", circuit), ("fake IBM", circuit_hw), ("real IBM", circuit_ibm)]:
    print(f"{name:9s}: {evaluate(circ, w, b, X_sub, y_sub):.3f}")

ibm_kingston
ideal    : 0.800
fake IBM : 0.700
real IBM : 0.725
